# 05 - Mental Model Comparison (Controlled A/B)

This notebook runs a controlled single-student comparison using the same student, same problem set, and same KC vocabulary:
- Condition A: Curriculum-Aware prompt (no mental model)
- Condition B: Curriculum-Aware prompt + student mental model payload

Target student: 14359 (average cluster medoid).

In [1]:
import json
import time
from pathlib import Path

import pandas as pd
from google.genai import types

ROOT = Path.cwd()
if not (ROOT / 'lib').exists() and (ROOT.parent / 'lib').exists():
    ROOT = ROOT.parent

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib.experiment_utils import create_client, load_best_attempts_df, save_results
from lib.llm_batch_analyzer import format_submissions, clean_json_response
from lib.mental_model import load_skill_map, calculate_student_profile, build_prerequisite_graph, get_weak_skills, get_prereq_risk_chains, build_mental_model_payload
try:
    from lib.prompt_strategies import build_curriculum_aware_prompt, KC_TAGS
except ModuleNotFoundError:
    from lib.prompts import build_curriculum_aware_prompt, KC_TAGS
from utils.dataset import load_topics_json, load_problem_descriptions

MODEL_ID = 'gemini-2.5-flash'
TARGET_STUDENT_ID = 14359
NUM_PROBLEMS = 12
RANDOM_SEED = 42
SLEEP_SECONDS = 1.0

client = create_client()
print(f'Ready. Model={MODEL_ID}, Student={TARGET_STUDENT_ID}, Requested problems={NUM_PROBLEMS}')

Ready. Model=gemini-2.5-flash, Student=14359, Requested problems=12


In [2]:
best_attempts_df = load_best_attempts_df()
student_df = best_attempts_df[best_attempts_df['SubjectID'] == TARGET_STUDENT_ID].copy()
student_df = student_df.drop_duplicates(subset=['ProblemID']).copy()

if student_df.empty:
    raise ValueError(f'No submissions found for student {TARGET_STUDENT_ID}.')

score_max = float(student_df['Score'].max())
if score_max <= 1.0:
    student_df['ScorePct'] = student_df['Score'] * 100.0
else:
    student_df['ScorePct'] = student_df['Score']

available_n = len(student_df)
target_n = min(NUM_PROBLEMS, available_n)

if available_n < 10:
    print(f'Fallback: student has only {available_n} problems (<10). Using all available problems.')
    selected_df = student_df.sample(n=target_n, random_state=RANDOM_SEED).copy()
else:
    full_df = student_df[student_df['ScorePct'] >= 99.9].copy()
    partial_df = student_df[(student_df['ScorePct'] > 0.0) & (student_df['ScorePct'] < 99.9)].copy()
    zero_df = student_df[student_df['ScorePct'] <= 0.0].copy()

    base_quota = target_n // 3
    remainder = target_n - (base_quota * 3)
    quotas = {'full': base_quota, 'partial': base_quota, 'zero': base_quota}
    for key in ['partial', 'full', 'zero'][:remainder]:
        quotas[key] += 1

    picked = []
    bucket_map = {'full': full_df, 'partial': partial_df, 'zero': zero_df}
    for name, bucket in bucket_map.items():
        take_n = min(quotas[name], len(bucket))
        if take_n > 0:
            picked.append(bucket.sample(n=take_n, random_state=RANDOM_SEED))

    selected_df = pd.concat(picked, ignore_index=False).drop_duplicates(subset=['ProblemID']) if picked else pd.DataFrame(columns=student_df.columns)

    remaining_needed = target_n - len(selected_df)
    if remaining_needed > 0:
        remaining_pool = student_df[~student_df['ProblemID'].isin(selected_df['ProblemID'])].copy()
        if len(remaining_pool) > 0:
            topup = remaining_pool.sample(n=min(remaining_needed, len(remaining_pool)), random_state=RANDOM_SEED)
            selected_df = pd.concat([selected_df, topup], ignore_index=False)

selected_df = selected_df.sort_values(by=['ScorePct', 'ProblemID'], ascending=[False, True]).head(target_n).copy()
selected_df = selected_df.reset_index(drop=True)

print(f'Available problems for student {TARGET_STUDENT_ID}: {available_n}')
print(f'Selected problems: {len(selected_df)}')
print('Score bucket counts in selected set:')
print(selected_df.assign(bucket=selected_df['ScorePct'].apply(lambda x: 'full' if x >= 99.9 else ('zero' if x <= 0.0 else 'partial')))['bucket'].value_counts())
display(selected_df[['ProblemID', 'Score', 'ScorePct']])

Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows
Joined dataset: 191,584 rows
Best attempts: 15,375 rows (372 students, 50 problems)
Available problems for student 14359: 49
Selected problems: 12
Score bucket counts in selected set:
bucket
full       7
partial    5
Name: count, dtype: int64


,ProblemID,Score,ScorePct
0,13,1.000000,100.0000
1,24,1.000000,100.0000
2,37,1.000000,100.0000
3,56,1.000000,100.0000
4,57,1.000000,100.0000
5,232,1.000000,100.0000
6,235,1.000000,100.0000
7,108,0.684211,68.4211
8,32,0.545455,54.5455
9,107,0.545455,54.5455


In [3]:
skill_map, all_skills = load_skill_map()
student_profile = calculate_student_profile(TARGET_STUDENT_ID, best_attempts_df, skill_map, all_skills)
weak_skills = get_weak_skills(student_profile, threshold=0.6)
G = build_prerequisite_graph()
risk_chains = get_prereq_risk_chains(G, weak_skills, max_items=8)

mental_model = build_mental_model_payload(
    student_id=TARGET_STUDENT_ID,
    profile=student_profile,
    weak_skill_pairs=weak_skills,
    graph=G,
)

print(f'Profile skills tracked: {len(student_profile)}')
print(f'Weak skills identified: {len(weak_skills)}')
print('Top weak skills:')
for skill, score in weak_skills[:8]:
    print(f'  - {skill}: {score:.3f}')
print('')
print(f'Risk chains found: {len(risk_chains)}')

Profile skills tracked: 18
Weak skills identified: 2
Top weak skills:
  - StringConcat: 0.578
  - While: 0.582

Risk chains found: 1


In [4]:
topics = load_topics_json() or {}
problem_descriptions = load_problem_descriptions() or {}
selected_problem_ids = [int(pid) for pid in selected_df['ProblemID'].tolist()]

baseline_prompt = build_curriculum_aware_prompt(
    topics=topics,
    problems=problem_descriptions,
    focus_problem_ids=selected_problem_ids,
)

print(f'Baseline prompt prepared for {len(selected_problem_ids)} problems.')
print('Enriched prompt will be created in the helper cell using the same baseline prompt + mental model JSON payload.')

Baseline prompt prepared for 12 problems.
Enriched prompt will be created in the helper cell using the same baseline prompt + mental model JSON payload.


In [5]:
EXACT_KC_TAGS = [
    'If/Else', 'NestedIf', 'While', 'For', 'NestedFor',
    'Math+-*/', 'Math%', 'LogicAndNotOr', 'LogicCompareNum', 'LogicBoolean',
    'StringFormat', 'StringConcat', 'StringIndex', 'StringLen',
    'StringEqual', 'CharEqual', 'ArrayIndex', 'DefFunction'
]
VALID_KC_SET = set(EXACT_KC_TAGS)

if set(KC_TAGS) != VALID_KC_SET:
    print('Warning: imported KC_TAGS differs from the required exact KC vocabulary. Validation will use EXACT_KC_TAGS.')

prompts_df = pd.read_csv(ROOT / 'dataset/CodeWorkout/Problem_Prompts/problem_prompts.csv')

def add_mental_model_context(base_prompt: str, mental_model_payload: dict) -> str:
    context = json.dumps(mental_model_payload, indent=2)
    return (
        base_prompt
        + "\n\nAdditional Student Mental Model Context:\n"
        + context
        + "\n\nUse this context to better judge likely misconceptions and future risks."
    )

def run_one_call(submission_row, system_instruction: str):
    formatted_input = format_submissions([submission_row.to_dict()])
    t0 = time.time()
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.3,
                response_mime_type='application/json',
            ),
        )
        raw_text = response.text if response and response.text else '{}'
        parsed = json.loads(clean_json_response(raw_text))
        return parsed, round(time.time() - t0, 3), None
    except Exception as e:
        return None, round(time.time() - t0, 3), str(e)

def extract_expected_kc_tags(problem_id: int) -> list[str]:
    row = prompts_df[prompts_df['ProblemID'] == int(problem_id)]
    if row.empty:
        return []
    row = row.iloc[0]
    tags = []
    for tag in EXACT_KC_TAGS:
        value = row.get(tag, 0)
        if pd.notna(value) and float(value) == 1.0:
            tags.append(tag)
    return tags

def extract_kc_tags(output_obj) -> tuple[list[str], list[str]]:
    """Extract KC tags from Curriculum-Aware LLM output structure."""
    if not isinstance(output_obj, dict):
        return [], []

    valid_tags = set()
    invalid_tags = set()
    analysis_list = output_obj.get("student_analysis", [])
    if not isinstance(analysis_list, list):
        analysis_list = []

    for analysis in analysis_list:
        if not isinstance(analysis, dict): continue
        for gap in analysis.get("knowledge_gaps", []):
            tag = ""
            if isinstance(gap, dict):
                tag = gap.get("missing_concept", "")
            elif isinstance(gap, str):
                tag = gap
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

        for pred in analysis.get("future_predictions", []):
            tag = ""
            if isinstance(pred, dict):
                tag = pred.get("at_risk_topic", "")
            elif isinstance(pred, str):
                tag = pred
            tag = tag.strip()
            if tag:
                if tag in VALID_KC_SET:
                    valid_tags.add(tag)
                else:
                    invalid_tags.add(tag)

    for key in ["knowledge_gaps", "future_predictions"]:
        val = output_obj.get(key, [])
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str):
                    item = item.strip()
                    if item in VALID_KC_SET:
                        valid_tags.add(item)
                    elif item:
                        invalid_tags.add(item)

    return sorted(valid_tags), sorted(invalid_tags)

def calculate_overlap(predicted_tags: list[str], expected_tags: list[str]) -> dict:
    pred_set = set(predicted_tags)
    exp_set = set(expected_tags)
    overlap = pred_set & exp_set
    precision = (len(overlap) / len(pred_set)) if pred_set else 0.0
    recall = (len(overlap) / len(exp_set)) if exp_set else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        'pred_count': len(pred_set),
        'expected_count': len(exp_set),
        'overlap_count': len(overlap),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
    }

enriched_prompt = add_mental_model_context(baseline_prompt, mental_model)
print('Helpers ready. Enriched prompt prepared.')

Helpers ready. Enriched prompt prepared.


In [6]:
weak_skill_names = [s[0] for s in weak_skills]
rows = []
total = len(selected_df)

for i, (_, sub) in enumerate(selected_df.iterrows(), start=1):
    pid = int(sub['ProblemID'])
    score_pct = float(sub['ScorePct'])
    expected_tags = extract_expected_kc_tags(pid)

    baseline_out, baseline_time, baseline_err = run_one_call(sub, baseline_prompt)
    enriched_out, enriched_time, enriched_err = run_one_call(sub, enriched_prompt)

    baseline_tags, baseline_invalid = extract_kc_tags(baseline_out if baseline_err is None else {})
    enriched_tags, enriched_invalid = extract_kc_tags(enriched_out if enriched_err is None else {})

    baseline_gap_count = 0
    enriched_gap_count = 0
    try:
        analysis = baseline_out.get("student_analysis", [{}]) if isinstance(baseline_out, dict) else [{}]
        baseline_gap_count = len(analysis[0].get("knowledge_gaps", [])) if isinstance(analysis, list) and len(analysis) > 0 else 0
    except: pass
    try:
        analysis = enriched_out.get("student_analysis", [{}]) if isinstance(enriched_out, dict) else [{}]
        enriched_gap_count = len(analysis[0].get("knowledge_gaps", [])) if isinstance(analysis, list) and len(analysis) > 0 else 0
    except: pass

    baseline_weak_overlap = calculate_overlap(baseline_tags, weak_skill_names)
    enriched_weak_overlap = calculate_overlap(enriched_tags, weak_skill_names)
    baseline_relevance = calculate_overlap(baseline_tags, expected_tags)
    enriched_relevance = calculate_overlap(enriched_tags, expected_tags)

    is_perfect = score_pct >= 99.9
    baseline_perfect_correct = (baseline_gap_count == 0) if is_perfect else None
    enriched_perfect_correct = (enriched_gap_count == 0) if is_perfect else None

    rows.append({
        'SubjectID': int(sub['SubjectID']),
        'ProblemID': pid,
        'Score': float(sub['Score']),
        'ScorePct': score_pct,
        'IsPerfect': is_perfect,
        'Expected_KCTags': expected_tags,
        'WeakSkills': weak_skill_names,
        'Baseline_GAP_Count': baseline_gap_count,
        'Enriched_GAP_Count': enriched_gap_count,
        'Baseline_WeakOverlap_F1': baseline_weak_overlap['f1'],
        'Enriched_WeakOverlap_F1': enriched_weak_overlap['f1'],
        'Baseline_KCTags': baseline_tags,
        'Enriched_KCTags': enriched_tags,
        'Baseline_TimeSec': baseline_time,
        'Enriched_TimeSec': enriched_time,
        'Baseline_PerfectCorrect': baseline_perfect_correct,
        'Enriched_PerfectCorrect': enriched_perfect_correct,
        'Baseline_Relevance_F1': baseline_relevance['f1'],
        'Enriched_Relevance_F1': enriched_relevance['f1'],
    })

    tag_status = "SAME" if baseline_tags == enriched_tags else "DIFF"
    print(f"[{i}/{total}] Problem {pid} | Score={score_pct:.1f}% | Gaps: B={baseline_gap_count} E={enriched_gap_count} [{tag_status}]")

    if SLEEP_SECONDS > 0: time.sleep(SLEEP_SECONDS)

comparison_df = pd.DataFrame(rows)
print(f"\nCompleted A/B runs: {len(comparison_df)} rows")
display(comparison_df[['ProblemID', 'ScorePct', 'Baseline_GAP_Count', 'Enriched_GAP_Count', 'Baseline_WeakOverlap_F1', 'Enriched_WeakOverlap_F1']])

[1/12] Problem 13 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[2/12] Problem 24 | Score=100.0% | Gaps: B=3 E=2 [DIFF]
[3/12] Problem 37 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[4/12] Problem 56 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[5/12] Problem 57 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[6/12] Problem 232 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[7/12] Problem 235 | Score=100.0% | Gaps: B=0 E=0 [SAME]
[8/12] Problem 108 | Score=68.4% | Gaps: B=2 E=3 [DIFF]
[9/12] Problem 32 | Score=54.5% | Gaps: B=2 E=3 [DIFF]
[10/12] Problem 107 | Score=54.5% | Gaps: B=2 E=1 [DIFF]
[11/12] Problem 40 | Score=38.5% | Gaps: B=4 E=3 [DIFF]
[12/12] Problem 34 | Score=14.3% | Gaps: B=3 E=3 [DIFF]

Completed A/B runs: 12 rows


,ProblemID,ScorePct,Baseline_GAP_Count,Enriched_GAP_Count,Baseline_WeakOverlap_F1,Enriched_WeakOverlap_F1
0,13,100.0000,0,0,0.0000,0.0000
1,24,100.0000,3,2,0.0000,0.0000
2,37,100.0000,0,0,0.0000,0.0000
3,56,100.0000,0,0,0.0000,0.0000
4,57,100.0000,0,0,0.0000,0.0000
5,232,100.0000,0,0,0.0000,0.0000
6,235,100.0000,0,0,0.0000,0.0000
7,108,68.4211,2,3,0.0000,0.0000
8,32,54.5455,2,3,0.0000,0.5714
9,107,54.5455,2,1,0.0000,0.4000


In [7]:
comparison_csv = f'mental_model_comparison_student_{TARGET_STUDENT_ID}.csv'
save_results(comparison_df, comparison_csv)
print(f'Saved: {comparison_csv}')
print('Columns:')
print(', '.join(comparison_df.columns))

Saved: mental_model_comparison_student_14359.csv
Columns:
SubjectID, ProblemID, Score, ScorePct, IsPerfect, Expected_KCTags, WeakSkills, Baseline_GAP_Count, Enriched_GAP_Count, Baseline_WeakOverlap_F1, Enriched_WeakOverlap_F1, Baseline_KCTags, Enriched_KCTags, Baseline_TimeSec, Enriched_TimeSec, Baseline_PerfectCorrect, Enriched_PerfectCorrect, Baseline_Relevance_F1, Enriched_Relevance_F1


In [8]:
print("=" * 70)
print("COMPARISON: Baseline vs Context-Enriched")
print("=" * 70)

# Split into perfect vs imperfect problems
perfect_df = comparison_df[comparison_df['IsPerfect'] == True]
imperfect_df = comparison_df[comparison_df['IsPerfect'] == False]

print(f"\n=== 100% Score Problems ({len(perfect_df)} problems) ===")
if len(perfect_df) > 0:
    for _, r in perfect_df.iterrows():
        b_correct = "PASS" if r['Baseline_PerfectCorrect'] else f"FAIL ({r['Baseline_GAP_Count']} gaps)"
        e_correct = "PASS" if r['Enriched_PerfectCorrect'] else f"FAIL ({r['Enriched_GAP_Count']} gaps)"
        print(f"  Problem {r['ProblemID']}: Baseline={b_correct}, Enriched={e_correct}")

print(f"\n=== Failing Problems ({len(imperfect_df)} problems) ===")
print(f"Student's weak skills: {weak_skill_names}")

if len(imperfect_df) > 0:
    for _, r in imperfect_df.iterrows():
        b_tags = ", ".join(r['Baseline_KCTags']) if r['Baseline_KCTags'] else "(none)"
        e_tags = ", ".join(r['Enriched_KCTags']) if r['Enriched_KCTags'] else "(none)"
        match = "SAME" if r['Baseline_KCTags'] == r['Enriched_KCTags'] else "DIFF"
        print(f"  Problem {r['ProblemID']}: Baseline={b_tags} | Enriched={e_tags} [{match}]")

print(f"\n=== Aggregate Metrics (Failing Problems) ===")
if len(imperfect_df) > 0:
    b_f1 = imperfect_df['Baseline_WeakOverlap_F1'].mean()
    e_f1 = imperfect_df['Enriched_WeakOverlap_F1'].mean()
    print(f"  Avg WeakOverlap F1: Baseline={b_f1:.3f}, Enriched={e_f1:.3f}, Delta={e_f1-b_f1:+.3f}")

COMPARISON: Baseline vs Context-Enriched

=== 100% Score Problems (7 problems) ===
  Problem 13: Baseline=PASS, Enriched=PASS
  Problem 24: Baseline=FAIL (3 gaps), Enriched=FAIL (2 gaps)
  Problem 37: Baseline=PASS, Enriched=PASS
  Problem 56: Baseline=PASS, Enriched=PASS
  Problem 57: Baseline=PASS, Enriched=PASS
  Problem 232: Baseline=PASS, Enriched=PASS
  Problem 235: Baseline=PASS, Enriched=PASS

=== Failing Problems (5 problems) ===
Student's weak skills: ['StringConcat', 'While']
  Problem 108: Baseline=ArrayIndex, For, LogicBoolean, NestedFor | Enriched=ArrayIndex, For, If/Else, LogicAndNotOr, NestedFor [DIFF]
  Problem 32: Baseline=For, If/Else, StringIndex | Enriched=For, If/Else, StringConcat, StringIndex, While [DIFF]
  Problem 107: Baseline=For, LogicAndNotOr, LogicBoolean, NestedFor | Enriched=For, NestedFor, While [DIFF]
  Problem 40: Baseline=For, LogicAndNotOr, NestedFor, StringEqual, StringIndex | Enriched=DefFunction, For, LogicBoolean, StringEqual, StringIndex [DIFF